# Atari RL Playground：强化学习与持续学习教程

本 Notebook 是统一教学入口：组织概念、参数、训练调用和结果阅读。算法、训练循环和指标计算由经过测试的 Python 文件实现。

| 案例 | 可用配置 | 教学目标 |
|---|---|---|
| 单任务 | DQN、PPO | 理解单个游戏的学习过程 |
| 联合多任务 | DQN、PPO | 观察共享表示与任务间干扰 |
| 普通顺序训练 | DQN、PPO | 观察新任务学习与旧任务遗忘 |
| 顺序训练 + EWC | PPO；DQN 作为效果待验证的扩展 | 观察重要参数保护的作用 |
| 顺序训练 + GPM | PPO | 观察累计子空间保护的作用 |

五类案例共九种可运行配置，其中 DQN + EWC 为扩展案例。持续学习效果比较的主线使用 PPO、PPO + EWC、PPO + GPM。

- `algorithms/`：DQN、PPO、EWC 与 GPM 实现。
- `environments/`、`training/`：环境处理、共享 PPO 运行时和完整回合评估。
- `scripts/train_single.py`、`scripts/train_multitask.py`、`scripts/train_continual.py`：三个训练入口。
- `scripts/run_experiments.py`：统一调度这些入口。

下面的训练调用都带 `--dry-run`，仅预览命令。本轮先整理案例；正式训练的预算、seed 和记录方案另行确定。运行环境应提前按 README 配置。


## 1. 环境准备

普通环境按照 README 安装：

```bash
python -m pip install -e .
```

请在启动 Notebook 之前完成安装。当前开发基线为 PyTorch 2.14、TorchRL 0.13.3 和 TensorDict 0.13；PyTorch wheel 自带 CUDA runtime，但运行机器仍需提供兼容的 NVIDIA 驱动。

## 2. 运行环境

下面只调用 Python 模块读取版本和设备信息。受限沙箱中出现 `Can't initialize NVML` 不等于 CUDA 计算不可用。

In [ ]:
from scripts.tutorial_examples import (
    run_dqn_update_demo, run_ewc_penalty_demo, run_gae_demo, runtime_summary,
)

runtime_summary()

## 3. DQN：从 transition 到 Bellman update

对于 transition $(s_t, a_t, r_t, s_{t+1}, d_t)$，本仓库使用目标网络构造

$$
y_t = r_t + \gamma (1-d_t) \max_a Q_{\text{target}}(s_{t+1}, a),
$$

并最小化

$$
\mathcal{L}_{\text{DQN}} = \mathbb{E}[(Q(s_t,a_t)-y_t)^2].
$$

下面的函数创建一个很小的合成 batch，并调用仓库中真实的 `DQNAgent.update()`。它只验证数据契约和更新路径，不代表 Atari 学习效果。

In [ ]:
run_dqn_update_demo()

## 4. PPO：GAE 与 clipped objective

优势估计使用

$$
\delta_t = r_t + \gamma(1-d_t)V(s_{t+1}) - V(s_t),
$$

$$
A_t = \delta_t + \gamma\lambda(1-d_t)A_{t+1}.
$$

PPO 再通过概率比 $r_t(\theta)$ 的 clipped objective 限制单次更新幅度：

$$
\mathcal{L}_{\text{clip}} = \mathbb{E}[\min(r_t A_t, \operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t)].
$$

下一单元格直接调用 `algorithms.ppo.generalized_advantage_estimate` 的固定输入示例。

In [ ]:
run_gae_demo()

## 5. EWC：稳定性正则项，而不是冲突求解器

任务结束后，PPO 使用策略负对数似然的逐样本平方梯度估计对角经验 Fisher；DQN 没有对应的策略似然，因此使用逐样本 TD-MSE 平方梯度作为重要性近似：

$$
F_i \approx \frac{1}{N}\sum_{n=1}^{N}\left(\frac{\partial \ell_n}{\partial \theta_i}\right)^2.
$$

后续任务增加二次惩罚：

$$
\mathcal{L}_{\text{EWC}} = \frac{\lambda}{2}\sum_i I_i(\theta_i-\theta_i^*)^2.
$$

其中 PPO 的 $I_i=F_i$ 只覆盖策略敏感的 backbone 和 actor，不覆盖 critic；DQN 的 $I_i$ 是 TD-gradient surrogate。下面只演示数学机制：权重位于已巩固参考点时 penalty 为零，移动重要权重后 penalty 增大。这个结果不能证明 EWC 改善了旧任务，也不能证明它解决了新旧任务的梯度冲突。

In [ ]:
run_ewc_penalty_demo()

## 6. 案例一：单任务 DQN / PPO

每次只学习一个游戏，用作算法入门以及后续多任务比较的参照。默认游戏为 Pong 和 Breakout；两种算法由同一个单任务入口选择。

下面预览完整训练命令。`--steps` 表示每个单任务运行的训练 transitions，`--games` 可以选择游戏。


In [ ]:
from scripts.run_experiments import main as run_experiment_matrix

run_experiment_matrix(("train", "single", "--algorithms", "dqn", "ppo", "--dry-run"))


## 7. 案例二：联合多任务 DQN / PPO

Pong、Breakout、SpaceInvaders 的数据在整个训练期间都可访问。各游戏交替参与更新，共享 backbone，使用独立输出 head。

观察一个共享网络能否兼顾多个游戏，以及任务之间的帮助或干扰。这里的 `--steps` 是所有游戏合计的 transitions；它与顺序训练中“每个游戏的预算”含义不同。比较前需要核对各游戏实际获得的训练量。联合训练结果不是保证更高的性能上界。


In [ ]:
run_experiment_matrix(("train", "multitask", "--algorithms", "dqn", "ppo", "--dry-run"))


## 8. 案例三：普通顺序训练 DQN / PPO

从随机初始化开始，依次训练 Pong → Breakout → SpaceInvaders。每个阶段只使用当前游戏的数据；DQN 的 replay buffer 也在新任务开始时重新创建。

完成每个阶段后评估所有已见任务，观察共享表示更新对旧任务的影响。先确认旧任务已学会，再讨论其遗忘程度。

统一预算使用 `--steps`；不同任务的预算可以用 `--task-steps` 按 `--games` 的顺序指定。


In [ ]:
run_experiment_matrix((
    "train", "continual", "--algorithms", "dqn", "ppo", "--method", "finetune", "--dry-run",
))


## 9. 案例四：PPO + EWC

训练入口、任务顺序和共享网络与普通顺序 PPO 相同。任务结束后估计策略经验 Fisher，后续更新对旧任务重要参数施加二次惩罚。

比较旧任务保留和新任务学习两方面。增大正则强度可能限制新任务学习，不应只展示旧任务的最终分数。`--ewc-lambda` 控制正则强度；`--use-ewc` 是直接训练脚本保留的兼容选项。


In [ ]:
run_experiment_matrix((
    "train", "continual", "--algorithms", "ppo", "--method", "ewc", "--dry-run",
))


## 10. 案例五：PPO + GPM，从初始化开始完整训练

此入口不需要预先准备 Pong checkpoint。它先正常训练 Pong，再按相同顺序学习 Breakout 和 SpaceInvaders。

每个任务结束后，冻结当前策略进行额外状态采样，建立并累积共享层的输入子空间。后续任务将共享层的实际 Adam 位移投影到旧子空间的正交补；偏置作为常数输入坐标处理。旧任务 head 不参与新任务优化。

默认能量阈值为 0.995；每个边界额外采集 32,768 transitions，并抽取 2,048 个状态。采样不执行 PPO 更新，其开销与训练预算分别记录。直接训练脚本提供 `--gpm-threshold`、`--gpm-samples` 和 `--gpm-collection-steps`。

观察阶段分数、旧任务保留和累计保护维数。完整流程不等于各游戏已经收敛；该实现投影的是 Adam 位移，不声称完整复现原论文。


In [ ]:
run_experiment_matrix((
    "train", "continual", "--algorithms", "ppo", "--method", "gpm", "--dry-run",
))


## 11. 配套入口：评估与结果展示

训练阶段 i 完成后，在已见任务 j 上评估，形成阶段×任务矩阵 R[i,j]；未训练任务的空项不应当作零分。

- 新任务学习：任务首次学完后的分数及学习过程。
- 旧任务保留：完成后续训练后，旧任务分数相对学成时和历史最佳值的变化。
- GPM 保护空间：各层累计维数，以及新任务更新的实际投影误差。

不同 Atari 游戏的原始奖励尺度不同，分别查看每个游戏。训练曲线使用训练奖励，正式评估使用确定性动作、原始奖励和完整回合。

下面预览 GPM checkpoint 的独立评估入口；将方法改为 `finetune` 或 `ewc` 可评估相应 PPO 模型。


In [ ]:
run_experiment_matrix((
    "evaluate", "continual", "--algorithms", "ppo", "--method", "gpm", "--dry-run",
))


In [ ]:
from pathlib import Path
from scripts.visualize_results import plot_continual_results

if list(Path("outputs").glob("**/continual_evaluation.json")):
    plot_continual_results("outputs")
else:
    print("No continual_evaluation.json found; run a continual experiment first.")

## 12. 扩展案例：DQN + EWC

该组合已有代码支持，但本仓库尚无其完整训练效果对照。当前 DQN 使用逐样本 TD 损失梯度平方估计参数重要性，属于重要性正则近似，不是 PPO 的策略经验 Fisher；默认边界样本量随 batch size 设置，默认取 32 条 transition。

此案例用于理解不同重要性定义。将它作为效果演示前，需要先验证旧任务确实学成，再在固定预算下比较普通 DQN 与 DQN + EWC。


In [ ]:
run_experiment_matrix((
    "train", "continual", "--algorithms", "dqn", "--method", "ewc", "--dry-run",
))


## 13. 性能附录

PPO 性能测量入口是 `scripts/benchmark_ppo_runtime.py`。同步/eager 是可读的默认路径，`--env-backend async --compile-ppo` 用于单任务和顺序 PPO 的优化路径。吞吐提升与学习效果需要分别验证。
